In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib import rc

# Configuración de animación
rc('animation', html='jshtml')
plt.rcParams['animation.embed_limit'] = 100.0

# =============================================================================
# 1. INTERFAZ DE USUARIO: MODIFIQUE AQUÍ LOS PARÁMETROS
# =============================================================================

# Defina cualquier número de tramos (Ej: [10.0, 12.0, 10.0] para 3 tramos)
TRAMOS_VIGA = [10, 12, 10]

# Cargas Vehiculares
EJES_CAMION = [
    {"P": 3.64, "offset": 0.0},   
    {"P": 14.56, "offset": -4.3},
    {"P": 14.56, "offset": -8.6}
]

EJES_TANDEM = [
    {"P": 11.2, "offset": 0.0},
    {"P": 11.2, "offset": -1.2}
]

# Carga Distribuida de Carril (Tnf/m)
W_CARRIL = 0.96

# Precisión del cálculo (0.05m = 5cm)
dx = 0.05 

# =============================================================================
# 2. MOTORES ANALÍTICOS (VEHÍCULOS Y CARRIL)
# =============================================================================
n_tramos = len(TRAMOS_VIGA)
n_sup = n_tramos + 1
L_total = sum(TRAMOS_VIGA)
apoyos_x = np.cumsum([0.0] + TRAMOS_VIGA)
nombres_apoyos = [chr(65+i) for i in range(n_sup)]
X = np.round(np.arange(0, L_total + dx, dx), 3)

def resolver_matriz_momentos(B):
    A = np.zeros((n_tramos-1, n_tramos-1))
    for i in range(n_tramos-1):
        A[i, i] = 2 * (TRAMOS_VIGA[i] + TRAMOS_VIGA[i+1])
        if i > 0: A[i, i-1] = TRAMOS_VIGA[i]
        if i < n_tramos-2: A[i, i+1] = TRAMOS_VIGA[i+1]
    return np.linalg.solve(A, B)

def calcular_fuerzas_vehiculo(xt, vehiculo):
    cargas_activas = [(c["P"], xt + c["offset"]) for c in vehiculo if 0 <= xt + c["offset"] <= L_total]
    M_sup = np.zeros(n_sup)
    
    if n_tramos > 1:
        B = np.zeros(n_tramos-1)
        for P, x_pos in cargas_activas:
            for k in range(n_tramos):
                if (apoyos_x[k] < x_pos <= apoyos_x[k+1]) or (k == 0 and x_pos == 0):
                    xl = x_pos - apoyos_x[k]; xr = TRAMOS_VIGA[k] - xl
                    if k < n_tramos - 1: B[k] -= P * xr * (TRAMOS_VIGA[k]**2 - xr**2) / TRAMOS_VIGA[k]
                    if k > 0: B[k-1] -= P * xl * (TRAMOS_VIGA[k]**2 - xl**2) / TRAMOS_VIGA[k]
        M_sup[1:-1] = resolver_matriz_momentos(B)
        
    R = np.zeros(n_sup)
    for k in range(n_tramos):
        V_k_R = (M_sup[k+1] - M_sup[k]) / TRAMOS_VIGA[k]
        V_k1_L = (M_sup[k+1] - M_sup[k]) / TRAMOS_VIGA[k]
        for P, x_pos in cargas_activas:
            if (apoyos_x[k] < x_pos <= apoyos_x[k+1]) or (k == 0 and x_pos == 0):
                xl = x_pos - apoyos_x[k]
                V_k_R += P * (TRAMOS_VIGA[k] - xl) / TRAMOS_VIGA[k]
                V_k1_L -= P * xl / TRAMOS_VIGA[k]
        R[k] += V_k_R
        R[k+1] -= V_k1_L
        
    V = np.zeros_like(X); M = np.zeros_like(X)
    for k, Rk in enumerate(R):
        V += Rk * (X >= apoyos_x[k] - 1e-7)
        M += Rk * np.maximum(0, X - apoyos_x[k])
    for P, x_pos in cargas_activas:
        V -= P * (X >= x_pos - 1e-7)
        M -= P * np.maximum(0, X - x_pos)
    M[0] = M[-1] = 0
    return V, M, cargas_activas, R

def calcular_fuerzas_carril(tramos_cargados):
    M_sup = np.zeros(n_sup)
    if n_tramos > 1:
        B = np.zeros(n_tramos-1)
        for k in range(n_tramos):
            if tramos_cargados[k]:
                wL3_4 = (W_CARRIL * TRAMOS_VIGA[k]**3) / 4.0
                if k < n_tramos - 1: B[k] -= wL3_4
                if k > 0: B[k-1] -= wL3_4
        M_sup[1:-1] = resolver_matriz_momentos(B)
        
    R = np.zeros(n_sup)
    for k in range(n_tramos):
        L_k = TRAMOS_VIGA[k]
        V_k_R = (M_sup[k+1] - M_sup[k]) / L_k
        V_k1_L = (M_sup[k+1] - M_sup[k]) / L_k
        if tramos_cargados[k]:
            V_k_R += W_CARRIL * L_k / 2.0
            V_k1_L -= W_CARRIL * L_k / 2.0
        R[k] += V_k_R
        R[k+1] -= V_k1_L
        
    V = np.zeros_like(X); M = np.zeros_like(X)
    for k, Rk in enumerate(R):
        V += Rk * (X >= apoyos_x[k] - 1e-7)
        M += Rk * np.maximum(0, X - apoyos_x[k])
        
    for k in range(n_tramos):
        if tramos_cargados[k]:
            st = apoyos_x[k]; ed = apoyos_x[k+1]
            in_span = (X > st) & (X <= ed)
            aft_span = (X > ed)
            V[in_span] -= W_CARRIL * (X[in_span] - st)
            V[aft_span] -= W_CARRIL * (ed - st)
            M[in_span] -= W_CARRIL * (X[in_span] - st)**2 / 2.0
            M[aft_span] -= W_CARRIL * (ed - st) * (X[aft_span] - (st + (ed - st)/2.0))
    M[0] = M[-1] = 0
    return V, M, R

# =============================================================================
# 3. ANÁLISIS Y GENERACIÓN DE MATRICES PUNTO A PUNTO
# =============================================================================
def init_env():
    return {"R_pos": {i: 0.0 for i in range(n_sup)}, "R_neg": {i: 0.0 for i in range(n_sup)}}

def evaluar_vehiculo(vehiculo):
    env_R = init_env()
    frames = np.arange(0.0, L_total + abs(min(c["offset"] for c in vehiculo)) + 1.0, dx)
    V_max_pt = np.full_like(X, -np.inf); V_min_pt = np.full_like(X, np.inf)
    M_max_pt = np.full_like(X, -np.inf); M_min_pt = np.full_like(X, np.inf)
    
    for xt in frames:
        V, M, _, R = calcular_fuerzas_vehiculo(xt, vehiculo)
        for i in range(n_sup):
            env_R["R_pos"][i] = max(env_R["R_pos"][i], R[i])
            env_R["R_neg"][i] = min(env_R["R_neg"][i], R[i])
            
        V_max_pt = np.maximum(V_max_pt, V); V_min_pt = np.minimum(V_min_pt, V)
        M_max_pt = np.maximum(M_max_pt, M); M_min_pt = np.minimum(M_min_pt, M)
        
    return env_R, frames, np.round(V_max_pt,3), np.round(V_min_pt,3), np.round(M_max_pt,3), np.round(M_min_pt,3)

casos_carril = []
casos_carril.append({"tramos": [(i % 2 == 0) for i in range(n_tramos)], "titulo": "Momento (+) Máximo - Tramos Impares"})
if n_tramos > 1:
    casos_carril.append({"tramos": [(i % 2 != 0) for i in range(n_tramos)], "titulo": "Momento (+) Máximo - Tramos Pares"})
for i in range(1, n_tramos):
    tramos = [False]*n_tramos; tramos[i-1] = True; tramos[i] = True
    casos_carril.append({"tramos": tramos, "titulo": f"Momento (-) Máximo - Apoyo {nombres_apoyos[i]}"})

env_R_carril = init_env()
V_car_max = np.full_like(X, -np.inf); V_car_min = np.full_like(X, np.inf)
M_car_max = np.full_like(X, -np.inf); M_car_min = np.full_like(X, np.inf)

for caso in casos_carril:
    V, M, R = calcular_fuerzas_carril(caso["tramos"])
    for i in range(n_sup):
        env_R_carril["R_pos"][i] = max(env_R_carril["R_pos"][i], R[i])
        env_R_carril["R_neg"][i] = min(env_R_carril["R_neg"][i], R[i])
        
    V_car_max = np.maximum(V_car_max, V); V_car_min = np.minimum(V_car_min, V)
    M_car_max = np.maximum(M_car_max, M); M_car_min = np.minimum(M_car_min, M)
    
V_car_max = np.round(V_car_max, 3); V_car_min = np.round(V_car_min, 3)
M_car_max = np.round(M_car_max, 3); M_car_min = np.round(M_car_min, 3)

print("Calculando respuestas dinámicas y estáticas...\n")
env_R_camion, frames_camion, V_cam_max, V_cam_min, M_cam_max, M_cam_min = evaluar_vehiculo(EJES_CAMION)
env_R_tandem, frames_tandem, V_tan_max, V_tan_min, M_tan_max, M_tan_min = evaluar_vehiculo(EJES_TANDEM)

V_cvt_max = np.maximum(V_cam_max, V_tan_max); V_cvt_min = np.minimum(V_cam_min, V_tan_min)
M_cvt_max = np.maximum(M_cam_max, M_tan_max); M_cvt_min = np.minimum(M_cam_min, M_tan_min)

R_cvt_pos = {i: max(env_R_camion["R_pos"][i], env_R_tandem["R_pos"][i]) for i in range(n_sup)}
R_cvt_neg = {i: min(env_R_camion["R_neg"][i], env_R_tandem["R_neg"][i]) for i in range(n_sup)}

V_comb_max = np.round(1.33 * V_cvt_max + V_car_max, 3)
V_comb_min = np.round(1.33 * V_cvt_min + V_car_min, 3)
M_comb_max = np.round(1.33 * M_cvt_max + M_car_max, 3)
M_comb_min = np.round(1.33 * M_cvt_min + M_car_min, 3)

R_comb_pos = {i: 1.33 * R_cvt_pos[i] + env_R_carril["R_pos"][i] for i in range(n_sup)}
R_comb_neg = {i: 1.33 * R_cvt_neg[i] + env_R_carril["R_neg"][i] for i in range(n_sup)}

# =============================================================================
# 4. MOTOR DE REPORTES CONECTADO DIRECTAMENTE A MATRICES
# =============================================================================
def generar_env_resumen(V_max_arr, V_min_arr, M_max_arr, M_min_arr, dict_R_pos, dict_R_neg):
    env = {
        "R_pos": dict_R_pos, "R_neg": dict_R_neg,
        "M_tram_pos": {i: 0.0 for i in range(n_tramos)}, "M_tram_neg": {i: 0.0 for i in range(n_tramos)},
        "M_apoy_pos": {i: 0.0 for i in range(1, n_tramos)}, "M_apoy_neg": {i: 0.0 for i in range(1, n_tramos)},
        "V_max": {i: 0.0 for i in range(n_tramos)}
    }
    for i in range(n_tramos):
        mask = (X >= apoyos_x[i]) & (X <= apoyos_x[i+1])
        if np.any(mask):
            env["M_tram_pos"][i] = np.max(M_max_arr[mask])
            env["V_max"][i] = max(np.max(np.abs(V_max_arr[mask])), np.max(np.abs(V_min_arr[mask])))
        
        x_c = apoyos_x[i] + (TRAMOS_VIGA[i] / 2.0)
        idx_c = np.argmin(np.abs(X - x_c))
        env["M_tram_neg"][i] = M_min_arr[idx_c]
        
    for i in range(1, n_tramos):
        idx = np.argmin(np.abs(X - apoyos_x[i]))
        env["M_apoy_pos"][i] = M_max_arr[idx]
        env["M_apoy_neg"][i] = M_min_arr[idx]
        
    return env

def imprimir_reporte(titulo, env):
    print("="*50 + f"\n      RESUMEN DE ENVOLVENTES ({titulo})\n" + "="*50)
    for i in range(n_sup): print(f"R_{nombres_apoyos[i]}_max : (+) {env['R_pos'][i]:>6.2f} Tnf  |  (-) {env['R_neg'][i]:>6.2f} Tnf")
    print("-" * 50)
    for i in range(n_tramos): print(f"M_{nombres_apoyos[i]}{nombres_apoyos[i+1]}_max : (+) {env['M_tram_pos'][i]:>6.2f} Tnf·m  |  (-) {env['M_tram_neg'][i]:>6.2f} Tnf·m")
    if n_tramos > 1:
        for i in range(1, n_tramos): print(f"M_{nombres_apoyos[i]}_max  : (+) {env['M_apoy_pos'][i]:>6.2f} Tnf·m  |  (-) {env['M_apoy_neg'][i]:>6.2f} Tnf·m")
    print("-" * 50)
    for i in range(n_tramos): print(f"V_{nombres_apoyos[i]}{nombres_apoyos[i+1]}_max :     {env['V_max'][i]:>6.2f} Tnf (Absoluto)")
    print("="*50 + "\n")

env_print_camion = generar_env_resumen(V_cam_max, V_cam_min, M_cam_max, M_cam_min, env_R_camion["R_pos"], env_R_camion["R_neg"])
env_print_tandem = generar_env_resumen(V_tan_max, V_tan_min, M_tan_max, M_tan_min, env_R_tandem["R_pos"], env_R_tandem["R_neg"])
env_print_veh_max = generar_env_resumen(V_cvt_max, V_cvt_min, M_cvt_max, M_cvt_min, R_cvt_pos, R_cvt_neg)
env_print_carril = generar_env_resumen(V_car_max, V_car_min, M_car_max, M_car_min, env_R_carril["R_pos"], env_R_carril["R_neg"])
env_print_comb = generar_env_resumen(V_comb_max, V_comb_min, M_comb_max, M_comb_min, R_comb_pos, R_comb_neg)

imprimir_reporte("CAMIÓN HS-20", env_print_camion)
imprimir_reporte("TÁNDEM", env_print_tandem)
imprimir_reporte("CAMIÓN VS TÁNDEM", env_print_veh_max)
imprimir_reporte("CARRIL", env_print_carril)
imprimir_reporte("1.33(Camión o Tandem) + Carril", env_print_comb)

# =============================================================================
# 5. EXPORTACIÓN A EXCEL 
# =============================================================================
print("Generando archivo Excel con coordenadas de gráficas...")
nombres_columnas = [
    "DFC Via (CAMIÓN HS-20) - Posición (m)", "DFC Via (CAMIÓN HS-20) - Máx (+) (Tnf)", "DFC Via (CAMIÓN HS-20) - Mín (-) (Tnf)",
    "DFC Via (TÁNDEM) - Posición (m)", "DFC Via (TÁNDEM) - Máx (+) (Tnf)", "DFC Via (TÁNDEM) - Mín (-) (Tnf)",
    "DFC Via (CAMIÓN VS TÁNDEM) - Posición (m)", "DFC Via (CAMIÓN VS TÁNDEM) - Máx (+) (Tnf)", "DFC Via (CAMIÓN VS TÁNDEM) - Mín (-) (Tnf)",
    "DFC Via (CARRIL) - Posición (m)", "DFC Via (CARRIL) - Máx (+) (Tnf)", "DFC Via (CARRIL) - Mín (-) (Tnf)",
    "DFC Via (1.33(Cam o Tan) + Carril) - Posición (m)", "DFC Via (1.33(Cam o Tan) + Carril) - Máx (+) (Tnf)", "DFC Via (1.33(Cam o Tan) + Carril) - Mín (-) (Tnf)",
    "DMF Via (CAMIÓN HS-20) - Posición (m)", "DMF Via (CAMIÓN HS-20) - Máx (+) (Tnf·m)", "DMF Via (CAMIÓN HS-20) - Mín (-) (Tnf·m)",
    "DMF Via (TÁNDEM) - Posición (m)", "DMF Via (TÁNDEM) - Máx (+) (Tnf·m)", "DMF Via (TÁNDEM) - Mín (-) (Tnf·m)",
    "DMF Via (CAMIÓN VS TÁNDEM) - Posición (m)", "DMF Via (CAMIÓN VS TÁNDEM) - Máx (+) (Tnf·m)", "DMF Via (CAMIÓN VS TÁNDEM) - Mín (-) (Tnf·m)",
    "DMF Via (CARRIL) - Posición (m)", "DMF Via (CARRIL) - Máx (+) (Tnf·m)", "DMF Via (CARRIL) - Mín (-) (Tnf·m)",
    "DMF Via (1.33(Cam o Tan) + Carril) - Posición (m)", "DMF Via (1.33(Cam o Tan) + Carril) - Máx (+) (Tnf·m)", "DMF Via (1.33(Cam o Tan) + Carril) - Mín (-) (Tnf·m)"
]

datos_matrices = np.column_stack((
    X, V_cam_max, V_cam_min,
    X, V_tan_max, V_tan_min,
    X, V_cvt_max, V_cvt_min,
    X, V_car_max, V_car_min,
    X, V_comb_max, V_comb_min,
    X, M_cam_max, M_cam_min,
    X, M_tan_max, M_tan_min,
    X, M_cvt_max, M_cvt_min,
    X, M_car_max, M_car_min,
    X, M_comb_max, M_comb_min
))

df_envolventes = pd.DataFrame(datos_matrices, columns=nombres_columnas)
nombre_excel = "Fuerzas Internas por Via.xlsx"
df_envolventes.to_excel(nombre_excel, index=False)
print(f"¡Exportación exitosa! Revisa el archivo '{nombre_excel}'.\n")

def config_ax(ax, y_lims, ylab):
    ax.set_xlim(-2.0, L_total + 2.0); ax.set_ylim(y_lims); ax.set_ylabel(ylab, fontweight='bold')
    ax.grid(True, axis='y', linestyle='--', alpha=0.6); ax.axhline(0, color='black', lw=1.5)
    for p in apoyos_x: ax.axvline(x=p, color='gray', linestyle='--', alpha=0.7)

# =============================================================================
# 6. CREACIÓN DE LA IMAGEN PNG (ENVOLVENTE FINAL COMBINADA)
# =============================================================================
print("Generando imagen estática de la Envolvente Combinada (PNG)...")
fig2, (ax_v, ax_m) = plt.subplots(2, 1, figsize=(12, 8))
fig2.suptitle("Envolvente de Carga Viva: 1.33(Camión o Tándem) + Carril", fontsize=16, fontweight='bold')

v_max_abs = max(np.max(V_comb_max), abs(np.min(V_comb_min)))
config_ax(ax_v, (-v_max_abs*1.2, v_max_abs*1.2), "Fuerza Cortante (Tnf)")
ax_v.plot(X, V_comb_max, 'b-', lw=1.5, label='Máx (+)')
ax_v.plot(X, V_comb_min, 'b--', lw=1.5, label='Mín (-)')
ax_v.fill_between(X, V_comb_min, V_comb_max, color='skyblue', alpha=0.4)
ax_v.legend(loc="upper right")

m_comb_max_val = np.max(M_comb_max)
m_comb_min_val = np.min(M_comb_min)
config_ax(ax_m, (m_comb_max_val*1.2 + 5, m_comb_min_val*1.2 - 5), "Momento Flector (Tnf·m)")
ax_m.set_xlabel("Distancia a lo largo de la estructura (m)", fontweight='bold')
ax_m.plot(X, M_comb_max, 'r-', lw=1.5, label='Máx (+)')
ax_m.plot(X, M_comb_min, 'r--', lw=1.5, label='Mín (-)')
ax_m.fill_between(X, M_comb_min, M_comb_max, color='salmon', alpha=0.4)
ax_m.legend(loc="upper right")

def anotar_picos_dfc(ax, Y_max, Y_min, color_txt):
    anotados_max = set(); anotados_min = set()
    for x_sup in apoyos_x:
        idx_sup = np.argmin(np.abs(X - x_sup))
        idx_ini = max(0, idx_sup - 2); idx_fin = min(len(X) - 1, idx_sup + 2)
        
        v_max = np.max(Y_max[idx_ini:idx_fin+1])
        idx_max = idx_ini + np.argmax(Y_max[idx_ini:idx_fin+1])
        x_max = X[idx_max]
        if v_max > 0.1 and x_max not in anotados_max:
            ax.plot(x_max, v_max, 'ko', markersize=4)
            ax.annotate(f"{v_max:.2f}", (x_max, v_max), textcoords="offset points", xytext=(0, 8), ha='center', fontsize=8, fontweight='bold', color=color_txt)
            anotados_max.add(x_max)
            
        v_min = np.min(Y_min[idx_ini:idx_fin+1])
        idx_min = idx_ini + np.argmin(Y_min[idx_ini:idx_fin+1])
        x_min = X[idx_min]
        if v_min < -0.1 and x_min not in anotados_min:
            ax.plot(x_min, v_min, 'ko', markersize=4)
            ax.annotate(f"{v_min:.2f}", (x_min, v_min), textcoords="offset points", xytext=(0, -12), ha='center', fontsize=8, fontweight='bold', color=color_txt)
            anotados_min.add(x_min)

def anotar_picos_dmf(ax, Y_max, Y_min, color_txt):
    anotados = set()
    for k in range(n_tramos):
        mask = (X >= apoyos_x[k]) & (X <= apoyos_x[k+1])
        idx_sub = mask.nonzero()[0]
        if len(idx_sub) == 0: continue
        
        id_max = idx_sub[np.argmax(Y_max[mask])]
        v_max = Y_max[id_max]; x_max = X[id_max]
        if abs(v_max) > 1e-3 and (x_max, 'max') not in anotados:
            ax.plot(x_max, v_max, 'ko', markersize=4)
            ax.annotate(f"{v_max:.2f}", (x_max, v_max), textcoords="offset points", xytext=(0, -12), ha='center', fontsize=8, fontweight='bold', color=color_txt)
            anotados.add((x_max, 'max'))
            
        id_min = idx_sub[np.argmin(Y_min[mask])]
        v_min = Y_min[id_min]; x_min = X[id_min]
        if abs(v_min) > 1e-3 and (x_min, 'min') not in anotados:
            ax.plot(x_min, v_min, 'ko', markersize=4)
            ax.annotate(f"{v_min:.2f}", (x_min, v_min), textcoords="offset points", xytext=(0, 8), ha='center', fontsize=8, fontweight='bold', color=color_txt)
            anotados.add((x_min, 'min'))

anotar_picos_dfc(ax_v, V_comb_max, V_comb_min, 'darkblue')
anotar_picos_dmf(ax_m, M_comb_max, M_comb_min, 'darkred')

plt.tight_layout()
plt.savefig("Envolvente de Carga Viva por Via.png", dpi=300)
print("¡Imagen PNG guardada como 'Envolvente de Carga Viva por Via.png'!\n")
plt.close(fig2)

# =============================================================================
# 7. ANIMACIÓN SECUENCIAL COMPLETA (GIF) - CON ENVOLVENTES ESTÁTICAS DE FONDO
# =============================================================================
print("Preparando renderizado del GIF animado...")
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [1.5, 2, 2]})

P_max = max(max(c["P"] for c in EJES_CAMION), max(c["P"] for c in EJES_TANDEM))
v_glob = max(max(env_print_veh_max["V_max"].values()), max(env_print_carril["V_max"].values()))
m_glob_max = max(max(env_print_veh_max["M_tram_pos"].values()), max(env_print_carril["M_tram_pos"].values()))
m_glob_min = min(min(env_print_veh_max["M_apoy_neg"].values()), min(env_print_carril["M_apoy_neg"].values())) if n_tramos > 1 else 0

ax1.set_xlim(-2.0, L_total + 2.0); ax1.set_ylim(-35, P_max + 20); ax1.axis('off')
ax1.plot([0, L_total], [0, 0], 'k-', lw=5)
ax1.plot(apoyos_x, np.full(n_sup, -2), 'k^', markersize=15)

txt_R = [ax1.text(apoyos_x[i], -15, f"Apoyo {nombres_apoyos[i]}", ha='center', fontweight='bold') for i in range(n_sup)]
txt_Rv = [ax1.text(apoyos_x[i], -25, "0.00", ha='center', color='darkblue', fontweight='bold') for i in range(n_sup)]

num_c = max(len(EJES_CAMION), len(EJES_TANDEM))
flechas = [ax1.annotate("", xy=(0,0), xytext=(0,0), arrowprops=dict(facecolor='red', shrink=0.05)) for _ in range(num_c)]
textos_p = [ax1.text(0, 0, "", color='red', fontweight='bold', ha='center') for _ in range(num_c)]
parches_carril = []

config_ax(ax2, (-v_glob*1.2 - 10, v_glob*1.2 + 10), "Fuerza Cortante (Tnf)")
config_ax(ax3, (m_glob_max*1.2 + 10, m_glob_min*1.2 - 10), "Momento Flector (Tnf·m)")
ax3.set_xlabel("Distancia a lo largo de la estructura (m)", fontweight='bold')

# Variables para guardar las envolventes estáticas del fondo (zorder inferior = 1)
line_V_env_max, = ax2.plot([], [], color='gray', lw=1.5, linestyle='--', zorder=1)
line_V_env_min, = ax2.plot([], [], color='gray', lw=1.5, linestyle='--', zorder=1)
fill_V_env = None

line_M_env_max, = ax3.plot([], [], color='gray', lw=1.5, linestyle='--', zorder=1)
line_M_env_min, = ax3.plot([], [], color='gray', lw=1.5, linestyle='--', zorder=1)
fill_M_env = None

# Variables para las curvas en movimiento (zorder superior = 2 y 3)
line_V, = ax2.plot([], [], 'b-', lw=2.5, zorder=3)
fill_V = ax2.fill_between([], [], color='skyblue', alpha=0.5, zorder=2)
dot_V, = ax2.plot([], [], 'mo', markersize=8, zorder=4)
txt_max_V = ax2.text(0, 0, "", color='purple', fontweight='bold', ha='center', va='bottom', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1), zorder=4)

line_M, = ax3.plot([], [], 'r-', lw=2.5, zorder=3)
fill_M = ax3.fill_between([], [], color='salmon', alpha=0.5, zorder=2)
dot_M, = ax3.plot([], [], 'mo', markersize=8, zorder=4)
txt_max_M = ax3.text(0, 0, "", color='purple', fontweight='bold', ha='center', va='top', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1), zorder=4)

escenas = []
for xt in frames_camion[::2]: escenas.append(('vehiculo', xt, EJES_CAMION, "Carga HL-93 Camión HS-20"))
for _ in range(10): escenas.append(('transicion', None, None, "Transición a Tándem..."))
for xt in frames_tandem[::2]: escenas.append(('vehiculo', xt, EJES_TANDEM, "Carga HL-93 Tandem"))
for _ in range(15): escenas.append(('transicion', None, None, "Evaluando Cargas Distribuidas..."))
for caso in casos_carril:
    for _ in range(60): 
        escenas.append(('carril', caso["tramos"], None, f"Carga de Carril HL-93: {caso['titulo']}"))

def update(frame_data):
    global fill_V, fill_M, fill_V_env, fill_M_env
    tipo, d1, d2, titulo = frame_data
    fig.suptitle(titulo, fontsize=16, fontweight='bold')
    
    for f in flechas: f.set_visible(False)
    for t in textos_p: t.set_text("")
    for p in parches_carril: p.remove()
    parches_carril.clear()
    
    # -------------------------------------------------------------------------
    # 1. ACTUALIZAR ENVOLVENTES ESTÁTICAS DE FONDO SEGÚN EL TIPO DE CARGA
    # -------------------------------------------------------------------------
    if "Camión" in titulo:
        v_env_max, v_env_min = V_cam_max, V_cam_min
        m_env_max, m_env_min = M_cam_max, M_cam_min
    elif "Tandem" in titulo:
        v_env_max, v_env_min = V_tan_max, V_tan_min
        m_env_max, m_env_min = M_tan_max, M_tan_min
    elif "Carril" in titulo:
        v_env_max, v_env_min = V_car_max, V_car_min
        m_env_max, m_env_min = M_car_max, M_car_min
    else:
        v_env_max, v_env_min = np.zeros_like(X), np.zeros_like(X)
        m_env_max, m_env_min = np.zeros_like(X), np.zeros_like(X)
        
    line_V_env_max.set_data(X, v_env_max)
    line_V_env_min.set_data(X, v_env_min)
    if fill_V_env is not None: fill_V_env.remove()
    fill_V_env = ax2.fill_between(X, v_env_min, v_env_max, color='lightgray', alpha=0.4, zorder=1)

    line_M_env_max.set_data(X, m_env_max)
    line_M_env_min.set_data(X, m_env_min)
    if fill_M_env is not None: fill_M_env.remove()
    fill_M_env = ax3.fill_between(X, m_env_min, m_env_max, color='lightgray', alpha=0.4, zorder=1)
    
    # -------------------------------------------------------------------------
    # 2. CÁLCULO DE GRÁFICOS DINÁMICOS EN MOVIMIENTO
    # -------------------------------------------------------------------------
    if tipo == 'transicion':
        for i in range(n_sup): txt_Rv[i].set_text("R = 0.00 Tnf")
        line_V.set_data([], []); fill_V.remove(); fill_V = ax2.fill_between([], [], color='skyblue', alpha=0.5, zorder=2)
        line_M.set_data([], []); fill_M.remove(); fill_M = ax3.fill_between([], [], color='salmon', alpha=0.5, zorder=2)
        dot_V.set_data([], []); txt_max_V.set_text(""); dot_M.set_data([], []); txt_max_M.set_text("")
        return line_V, line_M
        
    if tipo == 'vehiculo':
        V, M, cargas, R = calcular_fuerzas_vehiculo(d1, d2)
        for i, (P, x_p) in enumerate(cargas):
            flechas[i].set_visible(True); flechas[i].xy = (x_p, 0); flechas[i].set_position((x_p, P))
            textos_p[i].set_position((x_p, P + P_max*0.15)); textos_p[i].set_text(f"{int(P)} Tnf")
            
    elif tipo == 'carril':
        V, M, R = calcular_fuerzas_carril(d1)
        for k in range(n_tramos):
            if d1[k]:
                p1 = ax1.fill_between([apoyos_x[k], apoyos_x[k+1]], 0, 15, color='green', alpha=0.3)
                p2 = ax1.text(apoyos_x[k] + TRAMOS_VIGA[k]/2, 17, f"q = {W_CARRIL} Tnf/m", ha='center', color='darkgreen', fontweight='bold')
                parches_carril.extend([p1, p2])

    for i in range(n_sup): txt_Rv[i].set_text(f"R = {R[i]:.2f} Tnf")
    
    line_V.set_data(X, V)
    fill_V.remove(); fill_V = ax2.fill_between(X, 0, V, color='skyblue', alpha=0.8, zorder=2)
    if np.any(V != 0):
        idx = np.argmax(np.abs(V)); val = V[idx]; dot_V.set_data([X[idx]], [val])
        txt_max_V.set_position((X[idx], val + (v_glob * 0.1 if val >= 0 else -v_glob * 0.25))); txt_max_V.set_text(f"{val:.2f}")
    else: dot_V.set_data([], []); txt_max_V.set_text("")
        
    line_M.set_data(X, M)
    fill_M.remove(); fill_M = ax3.fill_between(X, 0, M, color='salmon', alpha=0.8, zorder=2)
    if np.any(M != 0):
        idx = np.argmax(np.abs(M)); val = M[idx]; dot_M.set_data([X[idx]], [val])
        txt_max_M.set_position((X[idx], val - (m_glob_max * 0.15 if val >= 0 else abs(m_glob_min) * 0.25))); txt_max_M.set_text(f"{val:.2f}")
    else: dot_M.set_data([], []); txt_max_M.set_text("")
    
    return line_V, line_M

print("Renderizando animación (Camión -> Tándem -> Casos de Carril Estáticos)...")
anim = FuncAnimation(fig, update, frames=escenas, interval=50, blit=False)

archivo = "Transito de Cargas en Puente.gif"
anim.save(archivo, writer="pillow", dpi=100)
print(f"¡Animación finalizada y guardada como '{archivo}'!")
plt.close(fig)

Calculando respuestas dinámicas y estáticas...

      RESUMEN DE ENVOLVENTES (CAMIÓN HS-20)
R_A_max : (+)  21.87 Tnf  |  (-)  -2.50 Tnf
R_B_max : (+)  27.96 Tnf  |  (-)  -2.93 Tnf
R_C_max : (+)  27.35 Tnf  |  (-)  -2.74 Tnf
R_D_max : (+)  21.30 Tnf  |  (-)  -2.56 Tnf
--------------------------------------------------
M_AB_max : (+)  36.64 Tnf·m  |  (-) -12.49 Tnf·m
M_BC_max : (+)  36.73 Tnf·m  |  (-)  -8.00 Tnf·m
M_CD_max : (+)  37.15 Tnf·m  |  (-) -12.82 Tnf·m
M_B_max  : (+)   6.00 Tnf·m  |  (-) -24.98 Tnf·m
M_C_max  : (+)   5.60 Tnf·m  |  (-) -25.64 Tnf·m
--------------------------------------------------
V_AB_max :      24.55 Tnf (Absoluto)
V_BC_max :      24.68 Tnf (Absoluto)
V_CD_max :      24.68 Tnf (Absoluto)

      RESUMEN DE ENVOLVENTES (TÁNDEM)
R_A_max : (+)  20.51 Tnf  |  (-)  -2.29 Tnf
R_B_max : (+)  21.73 Tnf  |  (-)  -2.78 Tnf
R_C_max : (+)  21.73 Tnf  |  (-)  -2.78 Tnf
R_D_max : (+)  20.51 Tnf  |  (-)  -2.29 Tnf
--------------------------------------------------
M_AB_max